Configuracion inicial, preparacion de datos:

In [12]:
!pip install fastparquet

   ---------------------------------------- 0.0/701.7 kB ? eta -:--:--
   -------------- ------------------------- 262.1/701.7 kB ? eta -:--:--
   ---------------------------------------- 701.7/701.7 kB 2.6 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 18.6 MB/s  0:00:00

   ---------------------------------------- 0/3 [fsspec]
   ------------- -------------------------- 1/3 [cramjam]
   -------------------------- ------------- 2/3 [fastparquet]
   ---------------------------------------- 3/3 [fastparquet]




[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression # O DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Cargar el dataset optimizado (Capa Silver)
df = pd.read_parquet('../data/processed/london_crime_cleaned.parquet', engine='fastparquet')

# 2. Separar Variables Predictoras (X) y Variable Objetivo (y)
X = df[['borough', 'major_category', 'year', 'month']]
y = df['value']

# 3. Transformación de Variables Categóricas (OneHotEncoding)
# Esto convierte el texto de comunas y categorías en columnas binarias (0 y 1)
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False), ['borough', 'major_category'])
    ],
    remainder='passthrough' # Mantiene 'year' y 'month' tal cual
)

X_processed = preprocessor.fit_transform(X)

# 4. División del conjunto en Entrenamiento (Train) y Prueba (Test)
# Usamos un 80% para entrenar y un 20% para evaluar el modelo (estándar de la rúbrica)
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

print(f"Datos de entrenamiento: {X_train.shape[0]} filas")
print(f"Datos de prueba: {X_test.shape[0]} filas")

Datos de entrenamiento: 60249 filas
Datos de prueba: 15063 filas


Entrenamiento de modelo base:

In [14]:
#inicio de modelo base
model =  LinearRegression()

#entrenar el modelo con los datos de entrenamiento
print("Entrenando el modelo...")
model.fit(X_train, y_train)
print("Modelo entrenado con éxito.")

Entrenando el modelo...
Modelo entrenado con éxito.


In [16]:
print(f"Intercepto del modelo (Sesgo base): {model.intercept_:.2f} delitos")

Intercepto del modelo (Sesgo base): 120.74 delitos


In [17]:
# 1. Extraer los nombres de las columnas transformadas
cat_encoder = preprocessor.named_transformers_['cat']
encoded_features = list(cat_encoder.get_feature_names_out(['borough', 'major_category']))

# 2. Sumar las variables que pasaron directo ('year' y 'month')
all_features = encoded_features + ['year', 'month']

pesos_ia = pd.DataFrame({
    'Variable': all_features,
    'Coeficiente (Impacto)': model.coef_
})

# Ordenar de mayor a menor impacto
pesos_ia = pesos_ia.sort_values(by='Coeficiente (Impacto)', ascending=False)

print("\n=== TOP 5 VARIABLES QUE MÁS HACEN SUBIR EL CRIMEN EN EL MODELO ===")
print(pesos_ia.head(5).to_string(index=False))

print("\n=== TOP 5 VARIABLES QUE MÁS HACEN BAJAR EL CRIMEN EN EL MODELO ===")
print(pesos_ia.tail(5).to_string(index=False))


=== TOP 5 VARIABLES QUE MÁS HACEN SUBIR EL CRIMEN EN EL MODELO ===
           Variable  Coeficiente (Impacto)
borough_Westminster              92.411272
    borough_Lambeth              47.030247
  borough_Southwark              36.836364
     borough_Camden              35.980426
     borough_Newham              33.716100

=== TOP 5 VARIABLES QUE MÁS HACEN BAJAR EL CRIMEN EN EL MODELO ===
                                Variable  Coeficiente (Impacto)
                    major_category_drugs             -64.116431
                  major_category_robbery             -68.159080
major_category_other notifiable offences             -88.474794
          major_category_sexual offences            -105.534607
         major_category_fraud or forgery            -105.616415


evaluacion de modelo inicial

In [18]:
# Crear una tabla comparativa de muestra
comparativa = pd.DataFrame({
    'Valor Real': y_test.values[:10],
    'Predicción IA': np.round(y_pred[:10], 2),
    'Diferencia (Error)': np.round(y_test.values[:10] - y_pred[:10], 2)
})
print("\n=== MUESTRA DE LAS PRIMERAS 10 PREDICCIONES ===")
print(comparativa)


=== MUESTRA DE LAS PRIMERAS 10 PREDICCIONES ===
   Valor Real  Predicción IA  Diferencia (Error)
0          32          31.61                0.39
1           0          19.03              -19.03
2          20          41.95              -21.95
3           8          57.54              -49.54
4           4          29.98              -25.98
5           3          17.96              -14.96
6           0           1.43               -1.43
7          38          54.58              -16.58
8          11          69.66              -58.66
9          52         105.12              -53.12


In [15]:
# 1. Realizar predicciones sobre el conjunto de prueba
y_pred = model.predict(X_test)

# 2. Calcular las métricas principales exigidas por el logro RA3
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

# 3. Mostrar resultados
print("=== EVALUACIÓN DEL MODELO BASE ===")
print(f"Error Absoluto Medio (MAE): {mae:.2f} delitos")
print(f"Error Cuadrático Medio (MSE): {mse:.2f}")
print(f"Raíz del Error Cuadrático Medio (RMSE): {rmse:.2f} delitos")
print(f"Coeficiente de Determinación (R²): {r2:.4f}")

=== EVALUACIÓN DEL MODELO BASE ===
Error Absoluto Medio (MAE): 45.76 delitos
Error Cuadrático Medio (MSE): 5599.10
Raíz del Error Cuadrático Medio (RMSE): 74.83 delitos
Coeficiente de Determinación (R²): 0.2327


Error Absoluto Medio (MAE = 45.76 delitos): Esto significa que, en promedio, las predicciones del modelo base se equivocan por aproximadamente 46 delitos mensuales al estimar el volumen de criminalidad en una comuna (borough) para una categoría específica.

Raíz del Error Cuadrático Medio (RMSE = 74.83 delitos): Al ser el RMSE mayor que el MAE, nos indica que el dataset de Londres presenta ciertos meses o comunas con "peaks" de delincuencia muy altos (valores atípicos o outliers). Como el RMSE penaliza con mayor fuerza los errores grandes, es normal que suba a 75 delitos de desviación en esos casos extremos.

Coeficiente de Determinación ($R^2$ = 0.2327): El modelo actual tiene un valor de 23.27%. Esto significa que las variables de entrada utilizadas (borough, major_category, year, month) logran explicar casi una cuarta parte de la variabilidad total de los delitos en Londres.

Conclusiones Iniciales y Justificación del Desempeño

Al tratarse del primer prototipo base (Baseline) del ciclo de vida de IA, un $R^2$ del 23% es un resultado esperado y metodológicamente correcto. Esto se debe a que implementamos un algoritmo puramente lineal (Regresión Lineal) con parámetros básicos y sin optimización. La delincuencia es un fenómeno multivariable complejo que no se comporta de forma estrictamente lineal.